# NeuroPlex Expansion — Ingestion Pipeline

Orchestrates data ingestion from 22 external sources into Unity Catalog Delta tables.

**Architecture:**
```
Source API → Ingestor (rate-limited) → NormalizedRecord → Delta MERGE → neuroplex_{source} table
                                                                          ↓
                                                             MCP tool auto-registers via registry
```

**Target:** `<catalog>.<schema>.neuroplex_*`  
**Registry:** `ingestion/source_registry.py` (22 sources, 5 types)  
**Schedule:** Run as Lakeflow Job with per-source tasks

In [0]:
import sys, os
from pathlib import PurePosixPath

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()
p = PurePosixPath(notebook_path)
repo_root = str(p.parent.parent if p.parent.name == "ingestion" else p.parent)
sys.path.insert(0, repo_root)

from ingestion.source_registry import (
    SOURCES, SOURCE_MAP, SOURCES_BY_TYPE, SourceType, Priority,
    get_sources, get_target_tables, generate_all_tool_schemas,
    generate_all_ddl, TARGET_CATALOG, TARGET_SCHEMA,
)

print(f"\u2705 Registry loaded: {len(SOURCES)} sources")
print(f"   High priority: {len(get_sources(priority=Priority.HIGH))}")
print(f"   Medium priority: {len(get_sources(priority=Priority.MEDIUM))}")
print()
for st, sources in SOURCES_BY_TYPE.items():
    print(f"   {st.value:15s} [{len(sources)}]: {', '.join(s.name for s in sources)}")

In [0]:
%sql
-- Run the generated DDL for all source tables.
-- Each table has: common base columns + VARIANT payload + Delta CDC enabled.
-- Execute generate_all_ddl() from Python cell below, or run individual statements.

CREATE SCHEMA IF NOT EXISTS dhbl_discovery_us_dev.genesis_schema;

-- Example: OpenTargets table
CREATE TABLE IF NOT EXISTS dhbl_discovery_us_dev.genesis_schema.neuroplex_opentargets (
    record_id STRING NOT NULL COMMENT 'Unique record identifier',
    source_key STRING NOT NULL COMMENT 'Source registry key: opentargets',
    gene_symbol STRING COMMENT 'Primary gene symbol (UPPER CASE)',
    disease STRING COMMENT 'Associated disease/condition',
    drug STRING COMMENT 'Associated drug/compound',
    title STRING COMMENT 'Record title or name',
    summary STRING COMMENT 'Brief summary or abstract',
    payload VARIANT NOT NULL COMMENT 'Full source record as structured JSON',
    ingested_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    source_updated_at TIMESTAMP COMMENT 'Last update timestamp from source',
    api_version STRING COMMENT 'API version used for ingestion'
)
USING DELTA
COMMENT 'OpenTargets: Target-disease scores, druggability, and disease-gene evidence'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'neuroplex.source_type' = 'druggability',
    'neuroplex.priority' = 'high',
    'neuroplex.ingest_method' = 'graphql'
);

In [0]:
# Generate and display DDL for all 22 tables
ddl = generate_all_ddl()
print(f"Generated {len(SOURCES)} CREATE TABLE statements")
print(f"Target: {TARGET_CATALOG}.{TARGET_SCHEMA}.neuroplex_*")
print()
# Uncomment to execute all DDL:
# for statement in ddl.split(';'):
#     if statement.strip():
#         spark.sql(statement)

In [0]:
from ingestion.ingestors.opentargets import OpenTargetsIngestor
from ingestion.source_registry import SOURCE_MAP

# Initialize ingestor from registry config
ingestor = OpenTargetsIngestor(SOURCE_MAP["opentargets"])

# Ingest data for key Eisai targets
targets = ["HCRT", "HCRTR1", "HCRTR2", "PSEN1", "APP", "MAPT", "SOD1", "GRN", "TARDBP"]

results = []
for gene in targets:
    print(f"\u23F3 Ingesting {gene}...")
    try:
        summary = ingestor.run(gene=gene, limit=50)
        results.append(summary)
        print(f"   \u2705 {summary['records_written']} records, {summary['elapsed_seconds']}s")
    except Exception as e:
        print(f"   \u274C {gene}: {e}")
        results.append({"source": "opentargets", "gene": gene, "error": str(e)})

print(f"\n\u2500\u2500\u2500\u2500 Summary \u2500\u2500\u2500\u2500")
print(f"Targets processed: {len(targets)}")
print(f"Total records: {sum(r.get('records_written', 0) for r in results)}")
print(f"Errors: {sum(1 for r in results if 'error' in r)}")

In [0]:
from ingestion.ingestors.gnomad import GnomadIngestor
from ingestion.source_registry import SOURCE_MAP

# Initialize gnomAD ingestor
gnomad_ingestor = GnomadIngestor(SOURCE_MAP["gnomad"])

# Key NDD/orexin/neuroscience genes for constraint analysis
gnomad_targets = [
    # Orexin system
    "HCRT", "HCRTR1", "HCRTR2",
    # Alzheimer's / FTD
    "PSEN1", "PSEN2", "APP", "MAPT", "GRN", "TARDBP", "C9orf72",
    # ALS
    "SOD1", "FUS", "OPTN", "TBK1",
    # NDD risk genes (top TADA hits)
    "CHD8", "SCN2A", "SYNGAP1", "DYRK1A", "ADNP",
    # Sleep/circadian
    "CLOCK", "PER2", "CRY1", "BMAL1",
]

results = []
for gene in gnomad_targets:
    print(f"\u23F3 gnomAD: {gene}...")
    try:
        summary = gnomad_ingestor.run(gene=gene, include_variants=True, limit=30)
        results.append(summary)
        print(f"   \u2705 {summary['records_written']} records ({summary['elapsed_seconds']}s)")
    except Exception as e:
        print(f"   \u274C {gene}: {e}")
        results.append({"source": "gnomad", "gene": gene, "error": str(e)})

print(f"\n\u2500\u2500\u2500\u2500 gnomAD Summary \u2500\u2500\u2500\u2500")
print(f"Genes processed: {len(gnomad_targets)}")
print(f"Total records: {sum(r.get('records_written', 0) for r in results)}")
print(f"Errors: {sum(1 for r in results if 'error' in r)}")

In [0]:
# Display gene constraint scores ordered by LoF intolerance
df_constraint = spark.sql("""
    SELECT
        gene_symbol,
        title,
        payload:constraint.pLI::DOUBLE AS pLI,
        payload:constraint.oe_lof_upper::DOUBLE AS LOEUF,
        payload:constraint.mis_z::DOUBLE AS mis_z,
        payload:constraint.obs_lof::INT AS obs_lof,
        payload:constraint.exp_lof::DOUBLE AS exp_lof,
        summary
    FROM dhbl_discovery_us_dev.genesis_schema.neuroplex_gnomad
    WHERE source_key = 'gnomad'
      AND title LIKE '%Gene Constraint%'
    ORDER BY pLI DESC
""")
df_constraint.display()

In [0]:
# Verify the OpenTargets table was populated
df = spark.sql("""
    SELECT source_key, gene_symbol, COUNT(*) as records,
           COUNT(DISTINCT disease) as diseases,
           COUNT(DISTINCT drug) as drugs
    FROM dhbl_discovery_us_dev.genesis_schema.neuroplex_opentargets
    GROUP BY source_key, gene_symbol
    ORDER BY records DESC
""")
df.display()

In [0]:
import json

# Generate tool schemas for all sources
all_schemas = generate_all_tool_schemas()
print(f"Generated {len(all_schemas)} tool schemas for MCP registration")
print()

# Show first schema as example
print("Example (OpenTargets):")
ot_schema = next(s for s in all_schemas if s['function']['name'] == 'query_opentargets')
print(json.dumps(ot_schema, indent=2))

print("\n\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
print("To use in app.py, replace static TOOL_SCHEMAS with:")
print("  from ingestion.source_registry import generate_all_tool_schemas")
print("  TOOL_SCHEMAS = EXISTING_SCHEMAS + generate_all_tool_schemas()")

In [0]:
from ingestion.base_ingestor import query_source
from ingestion.source_registry import SOURCE_MAP, get_tool_dataset_map

# This is how app.py will call dynamically registered tools:
# 1. Tool schemas auto-generated from registry
# 2. Tool functions use generic query_source() against Delta tables
# 3. Tool filtering uses get_tool_dataset_map() for sidebar checkbox mapping

# Example: simulate an LLM tool call to query_opentargets
result = query_source("opentargets", query="HCRT", limit=5)
print("Tool call result preview:")
print(result[:500])

print("\n\u2500\u2500\u2500\u2500 Tool → Dataset Map (for sidebar filtering) \u2500\u2500\u2500\u2500")
tool_map = get_tool_dataset_map()
for tool, ds_type in sorted(tool_map.items()):
    print(f"  {tool:35s} → {ds_type}")